# Week 07 — Uplift and incrementality

**Goal.** Learn to predict incremental conversion rather than conversion, and evaluate it with Qini curves you wrote yourself.

**Deliverable.** T-learner, class-transformation and X-learner, compared on Qini.

**Rough shape of the week.** 2h reading (Gutierrez) · 5h building · 2h write-up.

---
### Ground rules (they apply every week)

1. **Beat a dumb baseline or it didn't happen.** Logistic regression or the global mean.
   Log the baseline in the same table as the fancy model.
2. **Split by time, never at random.** `split.time_split` — and call
   `split.check_no_leakage` so the assertion, not your memory, enforces it.
3. **Log every run** with `registry.log_result(...)`, including the ones that lost.
   The losing runs are what make the write-up honest.
4. **Write the finding down** in this week's `README.md` while it is fresh.

### Reading

PDFs are in `papers/` next to this notebook — see `papers/README.md`.

In [ ]:
import sys, warnings
sys.path.insert(0, "..")
warnings.filterwarnings("ignore", category=FutureWarning)

%load_ext autoreload
%autoreload 2

import numpy as np, pandas as pd, matplotlib.pyplot as plt
from adslab import data, metrics, plots, split, registry, encoders, calibration

plots.use_style()
pd.set_option("display.width", 140, "display.max_columns", 60)
print("harness ready")

## The distinction the whole industry gets wrong

A CVR model finds users **likely to convert**. An uplift model finds users **who convert
*because* you showed them the ad**. These are different people, and the overlap can be
small: your best CVR segment is often people who were going to buy anyway, where the ad's
incremental value is approximately zero and possibly negative.

This is a randomised experiment, so a random split is correct here — the one dataset in
the repo where that is true. Read the `load_uplift` docstring on why `exposure` is
poison as a feature.

In [ ]:
# frac= takes a RANDOM subsample. The file is grouped by treatment, so reading its
# head gives you 100% treated rows and an experiment with no control arm.
up = data.load_uplift(frac=0.2, seed=0)   # 13.98M rows is more than a laptop needs
print(f"{len(up):,} rows, treated={up.treatment.mean():.1%}")

t = up[up.treatment == 1].conversion.mean()
c = up[up.treatment == 0].conversion.mean()
print(f"conversion: treated={t:.4%} control={c:.4%}  ATE={t-c:+.4%}  lift={t/c-1:+.1%}")

X, y, w = up[data.UPLIFT_FEATURES].values, up.conversion.values, up.treatment.values

## 1. Implement Qini first

Before any model. You cannot tune what you cannot measure, and uplift's failure mode is
that everything looks fine on AUC.

Fill in `qini_curve` and `qini_auc` in `adslab/metrics.py`. The trap is in the docstring:
the control group must be rescaled by `n_treated / n_control`, and here that ratio is
about 5.7. Skip it and a null model looks brilliant.

`tests/test_harness.py::test_qini_handles_unequal_group_sizes` checks exactly that.

In [ ]:
!cd .. && python -m pytest tests -q -k qini

## 2. T-learner (two models)

One model on treated, one on control, uplift = difference of predictions. Simple, and it
has a real weakness worth stating: each model is fitted to minimise *its own* prediction
error, and the difference of two well-fitted models can be mostly noise when the true
uplift is small relative to the outcome. Here the ATE is a fraction of a percent, so this
matters a lot.

In [ ]:
# TODO

## 3. Class transformation

Define $Z = Y\cdot\mathbb{1}[W{=}1] + (1-Y)\cdot\mathbb{1}[W{=}0]$ and fit a single
model to $Z$. Under 50/50 randomisation, $2\Pr(Z{=}1|x)-1$ estimates the uplift directly.

**Our split is not 50/50** — it is ~85/15. Work out the propensity-weighted version
before you use it; the textbook formula silently assumes balance and will be biased here.
Deriving that correction is the most valuable twenty minutes of the week.

In [ ]:
# TODO: weighted class transformation with p(W=1) = up.treatment.mean()

## 4. X-learner

Built for exactly this situation: imbalanced groups where one arm has far more data.
Impute the treatment effect for each unit using the other arm's model, fit models to
those imputed effects, and combine them weighted by the propensity score. See
`papers/kunzel2017-metalearners.pdf` §3.

Predict: X-learner should beat T-learner *here specifically* because of the 85/15 split.
Write the prediction down before you run it.

In [ ]:
# TODO

## 5. Qini comparison and the operating point

All learners on one Qini plot. Then the practical question: **at what fraction of the
population targeted is incremental value maximised?** That number, not the AUUC, is what
a campaign manager would act on.

Also worth checking, and it makes the best slide of the week: take your top decile by
*predicted conversion* and your top decile by *predicted uplift*, and measure how much
they overlap. If it's low, you have shown the point of the week in one number.

In [ ]:
# fig, ax = plt.subplots()
# ... one Qini curve per learner + random diagonal
# print(plots.save(fig, 7, "qini_comparison"))

---
## Log the results

Every model you tried, including the baseline and including the failures. `notes` is the
one sentence you would say out loud about the run — future-you assembles the write-up
from these, so write it now while you still remember why the run mattered.

In [ ]:
# registry.log_result(
#     week=7,
#     model="lightgbm_hashed_2^18",
#     metrics=metrics.evaluate(y_test, p_test),
#     dataset="attribution",
#     params=dict(n_bits=18, num_leaves=63, lr=0.05),
#     notes="beats LR by 0.011 AUC; most of the gain is from cat3 x cat7 interactions",
# )

print(registry.to_markdown(week=7))

---
## Write it up

Open `README.md` in this folder and fill in the three sections. Keep it to a page.

- **What I built** — one paragraph, no code.
- **What the numbers say** — paste the table above; say which comparison is the honest one.
- **What surprised me** — the part worth reading. If nothing surprised you, you probably
  did not stress the model hard enough.

Then commit:

```bash
git add week07_* results/
git commit -m "week 07: <the finding, not the task>"
```